# PolicyPal — evaluation

Runs the curated Q/A set through each RAG approach and shows **every question, its answer, and the evaluation result**, plus aggregate quality and cost.

Everything is set in the config cell below **before the run**. With the `mock` provider it runs offline and free; with a real provider it makes real API calls — mind the cost (see the warning above section 4).

## 1 · Configure (all parameters are set here, before the run)

In [1]:
import os
from policypal.config import load_config

# ===== RUN PARAMETERS (edit before running) =====
PDF_PATH   = "OmniCorp_Handbook_Challenge V2.pdf"
CASES_PATH = "eval/handbook_cases.json"
APPROACHES = ["traditional", "pageindex", "agentic"]   # drop some to cut cost
JUDGE      = True     # LLM-judge faithfulness (extra calls; ignored under mock)
TOP_K      = 6         # retrieval depth for the recall table
MAX_CASES  = None      # e.g. 3 to test cheaply; None = all cases

# ===== MODEL / SAMPLING / EMBEDDING / RETRIEVAL / STORAGE / COST PARAMETERS =====
# Any field left out keeps its .env value. Uncomment to override before start.
cfg = load_config(
    # --- LLM model + sampling ---
    # provider="openrouter",                 # openrouter | anthropic | openai | mock
    # openrouter_model="openai/gpt-5.4-nano",
    # temperature=0.0,                        # blank/None = provider default (some
    #                                         #   reasoning models reject a custom value)
    # top_p=1.0,
    # seed=114514,                                # reproducible outputs (OpenAI/OpenRouter)
    # max_tokens=2000,                        # per-call default cap
    # --- embedding model ---
    # embedding_provider="openrouter",        # local | openai | openrouter
    # embedding_model="qwen/qwen3-embedding-8b",
    # embedding_dim=1536,                      # request a specific embedding size
    # --- hybrid keyword+vector retrieval ---
    # bm25_k1=1.5, bm25_b=0.75, keyword_stem=True,
    # hybrid_method="rrf", hybrid_alpha=0.5,   # "rrf" or "weighted"
    # --- storage + cost ---
    # store_backend="pgvector",               # auto | memory | pgvector
    # store_rebuild=False,
    # cost_input_per_1m=0.05, cost_output_per_1m=0.40,
)
model = getattr(cfg, cfg.provider + "_model", "mock") if cfg.provider != "mock" else "mock"
print("provider  :", cfg.provider, "| model:", model,
      "| temp:", cfg.temperature, "top_p:", cfg.top_p, "seed:", cfg.seed)
print("embeddings:", cfg.embedding_provider, "| store:", cfg.store_backend,
      "| hybrid:", cfg.hybrid_method, "(k1={}, b={})".format(cfg.bm25_k1, cfg.bm25_b))
print("cost/1M   : ${} in / ${} out".format(cfg.cost_input_per_1m, cfg.cost_output_per_1m))

provider  : openrouter | model: openai/gpt-5.4-nano | temp: 0.0 top_p: 1.0 seed: 114514
embeddings: openrouter | store: auto | hybrid: rrf (k1=1.5, b=0.75)
cost/1M   : $0.2 in / $1.25 out


## 2 · Build the pipeline

In [2]:
from policypal import PolicyPal
from policypal.evaluation import load_cases

pal = PolicyPal(PDF_PATH, cfg)
cases = load_cases(CASES_PATH)
if MAX_CASES:
    cases = cases[:MAX_CASES]

print("{} blocks -> {} chunks, {} tree nodes".format(
    len(pal.blocks), len(pal.chunks), len(pal.node_index)))
print("store: {} (embedding dim {})".format(type(pal.store).__name__, pal.embedder.dim))
print("cases: {} ({} answerable, {} adversarial)".format(
    len(cases), sum(c.answerable for c in cases), sum(not c.answerable for c in cases)))

376 blocks -> 27 chunks, 17 tree nodes
store: PgVectorStore (embedding dim 1536)
cases: 15 (10 answerable, 5 adversarial)


## 3 · Retrieval recall by method (no LLM calls)

In [8]:
from policypal.evaluation import evaluate_retrieval, render_retrieval

print(render_retrieval(evaluate_retrieval(pal.store, cases, k=TOP_K)))

retrieval recall@6 over 10 answerable cases:
  dense               1.00  ████████████████████
  keyword             1.00  ████████████████████
  hybrid-rrf          1.00  ████████████████████
  hybrid-weighted     1.00  ████████████████████


## 4 · Every question, answer, and evaluation result (takes MUCH LONGER time)

⚠️ Makes real LLM calls: `len(cases) × len(APPROACHES)` (agentic uses several per case). Trim `APPROACHES` or set `MAX_CASES` above to limit cost.

In [3]:
from IPython.display import Markdown, display
from policypal.evaluation import evaluate

report = evaluate(pal, cases, approaches=APPROACHES, judge=JUDGE)

def verdict(cr):
    bits = ["abstain ✓" if cr.abstention_ok else "abstain ✗"]
    if cr.correctness is not None:
        bits.append("correct ✓" if cr.correctness else "correct ✗")
    if cr.evidence_recall is not None:
        bits.append("recall ✓" if cr.evidence_recall else "recall ✗")
    bits.append("citations {}/{} valid".format(cr.citations_valid, cr.citations_total))
    if cr.faithfulness is not None:
        bits.append("faithful {:.2f}".format(cr.faithfulness))
    return " · ".join(bits)

for i, case in enumerate(cases):
    tag = "answerable" if case.answerable else "ADVERSARIAL (should abstain)"
    parts = ["### Q{}. {}".format(i + 1, case.question), "*{} — {}*".format(case.id, tag), ""]
    for a in APPROACHES:
        cr = report[a]["results"][i]
        parts.append("**{}** — {}".format(a, verdict(cr)))
        parts.append("```")
        parts.append(cr.answer.render() if cr.answer else "(no answer)")
        parts.append("```")
        parts.append("")
    display(Markdown(chr(10).join(parts)))

### Q1. How many days per week can I currently work remotely?
*remote-days-current — answerable*

**traditional** — abstain ✓ · correct ✓ · recall ✓ · citations 1/1 valid · faithful 1.00
```
✅ Remote work is limited to 1 day per week maximum, and this day must be a Friday; all other days require in-person attendance.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > URGENT AMENDMENT: Remote Work Update (June 2024), p.14] “Effective immediately, the Remote Work Policy (Section 1) is superseded by this amendment. Due to real
estate utilization requirements, remote work is now limited to 1 day per week maximum. This day must be a
Friday. All other days require in-person attendance.”
— tokens: 1,647 in + 116 out  ·  1 LLM call(s)  ·  2.41s  ·  est. cost $0.0005
```

**pageindex** — abstain ✓ · correct ✓ · recall ✓ · citations 1/1 valid · faithful 1.00
```
✅ Remote work is currently limited to 1 day per week maximum, and that day must be a Friday.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > URGENT AMENDMENT: Remote Work Update (June 2024), p.14] “Due to real estate utilization requirements, remote work is now limited to 1 day per week maximum. This day must be a Friday. All other days require in-person attendance.”
— tokens: 1,717 in + 142 out  ·  2 LLM call(s)  ·  2.48s  ·  est. cost $0.0005
```

**agentic** — abstain ✓ · correct ✓ · recall ✓ · citations 2/2 valid · faithful 1.00
```
✅ Currently, remote work is limited to a maximum of 1 day per week.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > URGENT AMENDMENT: Remote Work Update (June 2024), p.14] “remote work is now limited to 1 day per week maximum”
✅ That remote day must be a Friday; all other days require in-person attendance.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > URGENT AMENDMENT: Remote Work Update (June 2024), p.14] “This day must be a Friday. All other days require in-person attendance.”
— tokens: 5,086 in + 260 out  ·  4 LLM call(s)  ·  4.30s  ·  est. cost $0.0013
```


### Q2. Is there a stipend for remote work equipment, and how much is it?
*remote-stipend-amount — answerable*

**traditional** — abstain ✓ · correct ✓ · recall ✓ · citations 3/3 valid · faithful 1.00
```
✅ Yes. The handbook includes a Remote Work Equipment Stipend policy that provides reimbursement for equipment and software needed for remote work.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > 1. Remote Work & Flexibility, p.2] “As part of our organization's commitment to fostering a flexible and productive work environment, we are pleased to introduce the Remote Work Equipment Stipend policy.”
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > 1. Remote Work & Flexibility, p.2] “The stipend will be used to reimburse employees for the purchase of equipment and software necessary to perform their job duties”
✅ The stipend provides a maximum reimbursement amount of $1,000 per year.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > 1. Remote Work & Flexibility, p.2] “The stipend will be provided on an annual basis, with a maximum reimbursement amount of $1,000 per year”
— tokens: 1,672 in + 226 out  ·  1 LLM call(s)  ·  3.14s  ·  est. cost $0.0006
```

**pageindex** — abstain ✓ · correct ✓ · recall ✓ · citations 2/2 valid · faithful 1.00
```
✅ Yes. The handbook provides a Remote Work Equipment Stipend for eligible employees approved to work remotely on a regular basis.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > 1. Remote Work & Flexibility, pp.2-3] “The Remote Work Equipment Stipend policy. This policy is designed to provide employees with the necessary tools and resources to effectively perform their job duties while working remotely.”
✅ The stipend is provided annually with a maximum reimbursement amount of $1,000 per year.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > 1. Remote Work & Flexibility, pp.2-3] “* The stipend will be provided on an annual basis, with a maximum reimbursement amount of $1,000 per year”
— tokens: 1,725 in + 241 out  ·  2 LLM call(s)  ·  3.04s  ·  est. cost $0.0006
```

**agentic** — abstain ✓ · correct ✓ · recall ✓ · citations 2/2 valid · faithful 1.00
```
✅ Yes. The handbook includes a Remote Work Equipment Stipend that reimburses employees for remote-work equipment/software needed for their job duties.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > 1. Remote Work & Flexibility, pp.2-3] “we are pleased to introduce the Remote Work Equipment Stipend policy. This policy is designed to provide employees with the necessary tools and resources to effectively perform their job duties while working remotely.”
✅ Amount: up to $1,000 maximum reimbursement per year.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > 1. Remote Work & Flexibility, pp.2-3] “with a maximum reimbursement amount of $1,000 per year”
— tokens: 6,780 in + 308 out  ·  5 LLM call(s)  ·  5.57s  ·  est. cost $0.0017
```


### Q3. What are the password length requirements?
*password-length — answerable*

**traditional** — abstain ✓ · correct ✓ · recall ✓ · citations 1/1 valid · faithful 1.00
```
✅ All passwords must be exactly 14 characters long. Passwords shorter or longer than 14 characters will be rejected.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > IT Security: Password Complexity, p.14] “To ensure security, all passwords must be exactly 14 characters long. Passwords shorter or longer than 14
characters will be rejected.”
— tokens: 1,563 in + 83 out  ·  1 LLM call(s)  ·  1.61s  ·  est. cost $0.0004
```

**pageindex** — abstain ✓ · correct ✓ · recall ✓ · citations 1/1 valid · faithful 1.00
```
✅ Passwords must be exactly 14 characters long; any shorter or longer passwords will be rejected.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > IT Security: Password Complexity, p.14] “To ensure security, all passwords must be exactly 14 characters long. Passwords shorter or longer than 14
characters will be rejected.”
— tokens: 1,112 in + 129 out  ·  2 LLM call(s)  ·  1.94s  ·  est. cost $0.0004
```

**agentic** — abstain ✓ · correct ✓ · recall ✓ · citations 1/1 valid · faithful 1.00
```
✅ All OmniCorp passwords must be exactly 14 characters long; passwords shorter or longer than 14 characters will be rejected.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > IT Security: Password Complexity, p.14] “To ensure security, all passwords must be exactly 14 characters long. Passwords shorter or longer than 14
characters will be rejected.”
— tokens: 13,787 in + 220 out  ·  7 LLM call(s)  ·  7.68s  ·  est. cost $0.0030
```


### Q4. Is the office open on Christmas?
*christmas-office-status — answerable*

**traditional** — abstain ✓ · correct ✓ · recall ✓ · citations 1/1 valid · faithful 0.00
```
✅ Yes. The office is open on Christmas.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > APPENDIX B: Holiday Schedule, pp.13-14] “Dec 25      | Christmas         | Closed”
— tokens: 1,863 in + 50 out  ·  1 LLM call(s)  ·  1.54s  ·  est. cost $0.0004
```

**pageindex** — abstain ✓ · correct ✓ · recall ✓ · citations 1/1 valid · faithful 1.00
```
✅ Yes—Christmas is marked as “Closed,” so the office is not open on Christmas.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > APPENDIX B: Holiday Schedule, pp.13-14] “Dec 25      | Christmas         | Closed”
— tokens: 1,170 in + 99 out  ·  2 LLM call(s)  ·  2.15s  ·  est. cost $0.0004
```

**agentic** — abstain ✓ · correct ✓ · recall ✓ · citations 1/1 valid · faithful 1.00
```
✅ No—OmniCorp’s office is closed on Christmas (Dec 25).
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > APPENDIX B: Holiday Schedule, pp.13-14] “Dec 25      | Christmas         | Closed”
— tokens: 13,039 in + 200 out  ·  7 LLM call(s)  ·  7.67s  ·  est. cost $0.0029
```


### Q5. What is the office status on Independence Day (July 4)?
*july4-office-status — answerable*

**traditional** — abstain ✓ · correct ✓ · recall ✓ · citations 1/1 valid · faithful 1.00
```
✅ On Independence Day (Jul 04), the office status is Open (Voluntary).
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > APPENDIX B: Holiday Schedule, pp.13-14] “Jul 04      | Independence Day  | Open (Voluntary)”
— tokens: 1,849 in + 64 out  ·  1 LLM call(s)  ·  3.01s  ·  est. cost $0.0004
```

**pageindex** — abstain ✓ · correct ✓ · recall ✓ · citations 1/1 valid · faithful 1.00
```
⚠️ UNVERIFIED On Independence Day (July 4), the office status is Open (Voluntary).
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > APPENDIX B: Holiday Schedule, pp.13-14] “Jul 04      | Independence Day  | Open (Voluntary)”
— tokens: 1,182 in + 103 out  ·  2 LLM call(s)  ·  1.84s  ·  est. cost $0.0004
```

**agentic** — abstain ✓ · correct ✓ · recall ✓ · citations 1/1 valid · faithful 1.00
```
⚠️ UNVERIFIED On Independence Day (July 4), the office status is **Open (Voluntary)**.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > APPENDIX B: Holiday Schedule, pp.13-14] “Jul 04      | Independence Day  | Open (Voluntary)”
— tokens: 15,006 in + 240 out  ·  7 LLM call(s)  ·  11.37s  ·  est. cost $0.0033
```


...

### Q11. How many days of bereavement leave am I entitled to?
*bereavement-leave — ADVERSARIAL (should abstain)*

**traditional** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 1,469 in + 14 out  ·  1 LLM call(s)  ·  1.53s  ·  est. cost $0.0003
```

**pageindex** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 900 in + 56 out  ·  1 LLM call(s)  ·  1.23s  ·  est. cost $0.0003
```

**agentic** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 7,096 in + 164 out  ·  4 LLM call(s)  ·  5.59s  ·  est. cost $0.0016
```


### Q12. What is the maternity and parental leave policy?
*parental-leave — ADVERSARIAL (should abstain)*

**traditional** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 1,902 in + 14 out  ·  1 LLM call(s)  ·  1.61s  ·  est. cost $0.0004
```

**pageindex** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 896 in + 84 out  ·  1 LLM call(s)  ·  1.33s  ·  est. cost $0.0003
```

**agentic** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 3,300 in + 107 out  ·  3 LLM call(s)  ·  4.86s  ·  est. cost $0.0008
```


### Q13. How does the 401(k) matching work?
*retirement-401k — ADVERSARIAL (should abstain)*

**traditional** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 1,694 in + 14 out  ·  1 LLM call(s)  ·  2.39s  ·  est. cost $0.0004
```

**pageindex** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 897 in + 97 out  ·  1 LLM call(s)  ·  1.43s  ·  est. cost $0.0003
```

**agentic** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 3,317 in + 99 out  ·  3 LLM call(s)  ·  3.17s  ·  est. cost $0.0008
```


### Q14. Is there tuition reimbursement for continuing education?
*tuition-reimbursement — ADVERSARIAL (should abstain)*

**traditional** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 2,119 in + 14 out  ·  1 LLM call(s)  ·  1.84s  ·  est. cost $0.0004
```

**pageindex** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 895 in + 77 out  ·  1 LLM call(s)  ·  1.23s  ·  est. cost $0.0003
```

**agentic** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 10,234 in + 132 out  ·  4 LLM call(s)  ·  4.82s  ·  est. cost $0.0022
```


### Q15. Can I bring my dog to the office?
*pet-office-policy — ADVERSARIAL (should abstain)*

**traditional** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 1,669 in + 14 out  ·  1 LLM call(s)  ·  1.23s  ·  est. cost $0.0004
```

**pageindex** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 896 in + 100 out  ·  1 LLM call(s)  ·  1.23s  ·  est. cost $0.0003
```

**agentic** — abstain ✓ · citations 0/0 valid
```
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 19,448 in + 206 out  ·  8 LLM call(s)  ·  8.30s  ·  est. cost $0.0041
```


## 5 · Aggregate metrics + cost

In [4]:
import pandas as pd
from policypal.evaluation import render_report

metrics = ["abstention_accuracy", "answer_correctness", "evidence_recall",
           "citation_validity", "faithfulness", "tokens_per_query",
           "cost_per_query", "total_cost", "latency_per_query", "total_latency"]
df = pd.DataFrame({a: {m: report[a].get(m) for m in metrics} for a in APPROACHES})
display(df.style.format(precision=4, na_rep="—"))

print(render_report(report))

,traditional,pageindex,agentic
abstention_accuracy,1.0000,0.9333,1.0000
answer_correctness,1.0000,0.9000,1.0000
evidence_recall,1.0000,0.9000,1.0000
citation_validity,1.0000,1.0000,1.0000
faithfulness,0.9000,1.0000,1.0000
tokens_per_query,1839.4667,1357.8000,9663.9333
cost_per_query,0.0004,0.0004,0.0022
total_cost,0.0066,0.0061,0.0323
latency_per_query,2.1341,2.1449,6.6763
total_latency,32.0118,32.1741,100.1439


metric                traditional   pageindex     agentic       
----------------------------------------------------------------
abstention_accuracy     1.00          0.93          1.00        
answer_correctness      1.00          0.90          1.00        
evidence_recall         1.00          0.90          1.00        
citation_validity       1.00          1.00          1.00        
faithfulness            0.90          1.00          1.00        
----------------------------------------------------------------
tokens/query            1839          1358          9664        
cost/query ($)         0.00044       0.00041       0.00215      
latency/query (s)         2.13          2.14          6.68      
